# Image Captioning - Kaggle Training Run

Clones repo, trains with auto-resume, pushes best checkpoint to HF Hub,
runs pycocoevalcap on COCO val2017.

## 1. Setup - repo, env, and data mounts

In [ ]:
import os, subprocess, sys

REPO_URL = "https://github.com/MohamedShakshak/Image-Captioning.git"
CONFIG_PATH = "configs/default.yaml"

try:
    cwd = os.getcwd()
except FileNotFoundError:
    os.chdir("/kaggle/working")
    os.chdir("/kaggle/working")

if os.path.isdir("Image-Captioning"):
    %cd Image-Captioning
else:
    !git clone --depth 1 {REPO_URL}
    %cd Image-Captioning

!pip install -e . -q
!pip install -e .[app] -q
!pip install 'huggingface_hub<0.24' -q  # pin to avoid Xet bug
!pip install git+https://github.com/salaniz/pycocoevalcap.git -q

_src = os.path.join(os.getcwd(), "src")
if _src not in sys.path:
    sys.path.insert(0, _src)

import torch; print(f"Torch: {torch.__version__}, CUDA: {torch.cuda.is_available()}")

In [ ]:
COCO_ROOT = "/kaggle/input/datasets/awsaf49/coco-2017-dataset/coco2017"
ANNO_TRAIN = f"{COCO_ROOT}/annotations/captions_train2017.json"
ANNO_VAL = f"{COCO_ROOT}/annotations/captions_val2017.json"
FEATURES_DIR = "/kaggle/input/datasets/mohamedshakshak/resnet50"
CHECKPOINT_DIR = "/kaggle/working/checkpoints"
VOCAB_PATH = f"{FEATURES_DIR}/vocab.json"

for p in [COCO_ROOT, ANNO_TRAIN, ANNO_VAL, FEATURES_DIR]:
    assert os.path.exists(p), f"Missing mount: {p}"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

## 2. Training

In [ ]:
import sys
sys.argv = [
    "train.py", "--config", CONFIG_PATH,
    f"data.coco-root={COCO_ROOT}",
    f"data.annotation-train={ANNO_TRAIN}",
    f"data.annotation-val={ANNO_VAL}",
    f"data.features-dir={FEATURES_DIR}",
    f"data.vocab-path={VOCAB_PATH}",
    f"checkpoint.dir={CHECKPOINT_DIR}",
    "checkpoint.save-latest=true",
    "checkpoint.save-best=true",
    "train.amp=false",
]

from train import train as run_training
from config import parse_args
cfg, _ = parse_args()
run_training(cfg)

## 3. Evaluation on COCO val2017

In [ ]:
import sys
sys.argv = [
    "evaluate.py", "--config", CONFIG_PATH,
    f"data.coco-root={COCO_ROOT}",
    f"data.annotation-train={ANNO_TRAIN}",
    f"data.annotation-val={ANNO_VAL}",
    f"data.features-dir={FEATURES_DIR}",
    f"data.vocab-path={VOCAB_PATH}",
    f"checkpoint.dir={CHECKPOINT_DIR}",
    "eval.full-metrics=true",
    "eval.beam-size=3",
]

from evaluate import evaluate as run_eval
from config import parse_args
cfg, _ = parse_args()
run_eval(cfg)

## 4. Results & Upload

In [ ]:
import shutil
METRICS_PATH = f"{CHECKPOINT_DIR}/metrics.json"
if os.path.exists(METRICS_PATH):
    !python scripts/plot_curves.py --metrics "{METRICS_PATH}" --out /kaggle/working/curves.png
    shutil.copy(METRICS_PATH, "/kaggle/working/metrics.json")
else:
    print(f"metrics.json not found at {METRICS_PATH}")

for f in ["eval_out/preds.json", "eval_out/gts.json"]:
    if os.path.exists(f):
        shutil.copy(f, "/kaggle/working/")
        print(f"Saved /kaggle/working/{f}")

In [ ]:
HF_TOKEN = os.environ.get("HF_TOKEN")
HF_REPO = "MohamedShakshak/image-captioning-pytorch"
if HF_TOKEN:
    from huggingface_hub import HfApi
    api = HfApi(token=HF_TOKEN)
    api.create_repo(repo_id=HF_REPO, repo_type="model", exist_ok=True)
    for fn in ["best.pt", "vocab.json"]:
        p = f"{CHECKPOINT_DIR}/{fn}" if fn == "best.pt" else VOCAB_PATH
        if os.path.exists(p):
            api.upload_file(path_or_fileobj=p, path_in_repo=fn,
                            repo_id=HF_REPO, repo_type="model")
            print(f"Uploaded {fn}")
else:
    print("No HF_TOKEN - results in /kaggle/working/.")

## 5. Summary

- `/kaggle/working/checkpoints/best.pt` - best model
- `/kaggle/working/checkpoints/metrics.json` - per-epoch log
- `/kaggle/working/curves.png` - training curve
- `MohamedShakshak/image-captioning-pytorch` - HF Hub